# Functional specialization metrics: deep dive

This notebook documents the **three specialization metrics** used in the A1B2 analysis: how they are defined, what data they need, how they are computed, how to verify them, how to interpret the output, and what temporal detail we can get (over the training course).

## 1. Setup and overview

Run the cell below to import the modules that implement the metrics. No data load required for the documentation; verification cells may load one run if available.

In [1]:
import sys
import os
from pathlib import Path
import numpy as np

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.analysis.retraining_a1b2 import (
    retraining_specialization_scalar,
    metric_norm_acc,
    diff_metric,
    global_diff_metric,
)
from a1b2.analysis.correlations_a1b2 import (
    compute_correlation_metric_a1b2,
    get_correlation_a1b2,
    correlation_specialization_scalar,
    randperm_no_fixed,
)
print("Imports OK. Project root:", project_root)

Imports OK. Project root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular


### Overview of the three metrics

| Metric | What it measures | Data required | Output |
|--------|------------------|---------------|--------|
| **Retraining** | After training a 3-head probe (only M0, only M1, both) on frozen representations: how much does each module dominate decoding of feature 0 vs feature 1? | Trained wrapper + DataLoader (A1+B+A2) with `input`, `label_x`, `label_y`, `feature_probe` | Scalar in [0, 1]; high = one module for one feature, other for other |
| **Correlation** | When we fix the probed feature (summer/winter), within-module self-correlation of hidden activity; normalized by random-permutation baseline | `hiddens_per_module`, `probes`, `inputs` (or `stim_index`) from npz; flattened over phases | Same scalar formula; high = module–feature alignment |
| **Ablation** | Same as retraining but *without* training the probe: linear readout from only-M0 / only-M1 on frozen hiddens | Same as retraining (reuses trained probe from retraining step) | Same scalar; measures "linear separability" of features per module |

All three use the same **scalar formula** (Bená-style):  
`global_diff = |diff(metric_f0) - diff(metric_f1)| / 2` where `diff((a,b)) = (a-b)/(a+b)` and the metric pairs are (M0, M1) for feature 0 and (M0, M1) for feature 1.

---
## 2. Retraining specialization

### Definition

- **Idea**: Freeze the trained network (core + comms); replace the readout with a **3-head probe** (head 0: only module 0, head 1: only module 1, head 2: both). Train the probe with MSE on the **probed feature only** (summer or winter, from `feature_probe`). Then evaluate **regression accuracy** (1 − min(angular_error/π, 1)) for each head on each feature’s trials.
- **Quantities**: `acc[head_idx, feature_idx]` with shape (3, 2). We use only heads 0 and 1 (only M0, only M1). So:
  - `acc_M0_f0` = accuracy of “only M0” head on feature-0-probed trials  
  - `acc_M1_f0` = accuracy of “only M1” head on feature-0-probed trials  
  - `acc_M0_f1`, `acc_M1_f1` = same for feature 1.

### Formula

- Chance-correct and clip: `m = clip(acc - chance, 1e-5, 1)`.
- Per-feature imbalance: `diff_f0 = (m0_f0 - m1_f0) / (m0_f0 + m1_f0)`, `diff_f1 = (m0_f1 - m1_f1) / (m0_f1 + m1_f1)`.
- **Scalar**: `retraining_specialization = |diff_f0 - diff_f1| / 2` (in [0, 1]). High when one module is better for feature 0 and the other for feature 1 (or the opposite).

### Data collection

- **Source**: Script `03_functional_specialization.py` builds a single DataLoader = **ConcatDataset(A1, B, A2)** for one participant (from `get_datasets` + `CreateParticipantDataset`). Each batch must contain:
  - `input`: (batch, 12) stimulus
  - `label_x`, `label_y`: cos/sin of the **probed** feature’s angle (same as in training)
  - `feature_probe`: 0 or 1 (which feature is probed)
- **Training**: Probe is trained for 5 epochs (default) with AdamW on MSE over the probed feature slice only (cos/sin for that feature). All three heads are trained jointly on the same batches.
- **Evaluation**: Same loader; for each head we compute mean regression accuracy over trials where `feature_probe == 0` and where `feature_probe == 1`, giving `acc[3, 2]`. Only `acc[0,:]` and `acc[1,:]` are used for the scalar.

### Computation path (code)

1. `create_retraining_model_a1b2(wrapper, device)` — replace `community.readout` with `ProbeReadout` (3 heads with masks), freeze all params except readout.
2. `train_probe_readout_a1b2(wrapper, loader, condition, n_epochs=5, lr=1e-3)` — train probe on loader; loss = MSE on probed feature slice only, averaged over the 3 heads and batch.
3. `eval_probe_readout_a1b2(wrapper, loader)` → `acc` shape (3, 2).
4. `retraining_specialization_scalar(acc[0,0], acc[1,0], acc[0,1], acc[1,1])` → float.

### Verification: scalar formula with synthetic accs

Check that the scalar behaves as expected for extreme cases.

In [2]:
# Synthetic accuracies: (acc_M0_f0, acc_M1_f0, acc_M0_f1, acc_M1_f1)
# Perfect specialization: M0 good for f0, M1 good for f1
spec_high = retraining_specialization_scalar(0.9, 0.1, 0.1, 0.9)
# No specialization: both modules equal for both features
spec_low = retraining_specialization_scalar(0.5, 0.5, 0.5, 0.5)
# Reverse specialization: M0 good for f1, M1 good for f0 (still high)
spec_reverse = retraining_specialization_scalar(0.1, 0.9, 0.9, 0.1)
print("Perfect specialization (M0→f0, M1→f1):", round(spec_high, 4))
print("No specialization (equal):", round(spec_low, 4))
print("Reverse (M0→f1, M1→f0):", round(spec_reverse, 4))
print("Formula check: diff_f0 = (0.9-0.1)/(0.9+0.1)=0.8, diff_f1 = (0.1-0.9)/1.0=-0.8 → global = |0.8 - (-0.8)|/2 = 0.8")

Perfect specialization (M0→f0, M1→f1): 0.8
No specialization (equal): 0.0
Reverse (M0→f1, M1→f0): 0.8
Formula check: diff_f0 = (0.9-0.1)/(0.9+0.1)=0.8, diff_f1 = (0.1-0.9)/1.0=-0.8 → global = |0.8 - (-0.8)|/2 = 0.8


### Interpretation and temporal detail

- **Interpretation**: High retraining specialization means that, after training a linear probe on frozen hidden states, **one module’s activity predicts feature 0 well and the other predicts feature 1 well** (or the reverse). Low means both modules contribute similarly to both features. The value is in [0, 1]; 0 = no imbalance, 1 = maximum imbalance (one module perfect for one feature and useless for the other).
- **Temporal detail**: The pipeline currently produces **one number per participant** (post full training). To get specialization **over the training course** you would need to:
  - Save checkpoints (e.g. state after A1, after B, after A2),
  - For each checkpoint: load state → create probe → train probe on the same (or phase-matched) loader → evaluate → compute scalar.
  - The loader used in the script is always full A1+B+A2; for phase-resolved specialization you could instead build loaders per phase (e.g. A1 only, B only, A2 only) and report scalar per phase.

---
## 3. Correlation specialization

### Definition

- **Idea**: “Fixed-information” correlation (Bená-style): fix one factor (here: **feature**, i.e. probe 0 or 1), permute trial order within that subset. For each module, compute **within-module self-correlation**: corr(h[m], h[perm, m]) (Pearson by default). A **baseline** is the same correlation with a random permutation over *all* trials (no fixed factor). Normalize: `(corr_fix - base) / (1 - base)` per module so that 1 = same as fixed-factor stability, 0 = same as baseline.
- **Quantities**: For “fix feature 0”: subset trials with `probes == 0`, permute within; get correlation per module → `corr0` (length n_modules). Same for “fix feature 1” → `corr1`. Baseline: random permutation over all trials, no fixed points → `base`. Then `norm0 = (corr0 - base) / (1 - base)`, `norm1 = (corr1 - base) / (1 - base)` (with denom clipped to avoid division by zero).

### Formula

- **Scalar**: Same as retraining: `correlation_specialization = |diff(norm0[0], norm0[1]) - diff(norm1[0], norm1[1])| / 2`, i.e. global_diff_metric applied to (M0, M1) for fix-feature-0 and (M0, M1) for fix-feature-1. High when fixing feature 0 one module is more stable (higher norm) and the other less, and the pattern flips for feature 1.

### Data collection

- **Source**: From each participant’s **sim_<participant>.npz** (saved by the simulation script). Required keys:
  - `hiddens_per_module`: shape `(n_phase, n_trials, n_modules, hidden_size)` — hidden activations **split by module** (from `_split_hiddens_by_module` in schedule).
  - `probes`: shape `(n_phase, n_trials)` — which feature was probed (0 or 1) per trial.
  - `inputs` (or `stim_index`): for “fix stimulus” variant; for default “fix feature” only probes are used.
- **Flattening**: In `compute_correlation_metric_a1b2`, phase and trial dimensions are flattened so that `hiddens` is `(N, n_modules, hidden_size)` and `probes` is `(N,)`. So **all phases (A1, B, A2) are concatenated**; the metric is over the full schedule.
- **Randomness**: Baseline and fix-feature correlations are averaged over `n_samples=10` random permutations to reduce variance.

### Computation path (code)

1. Flatten `hiddens_per_module` and `probes` to (N, n_mod, h) and (N,).
2. For each of `n_samples` iterations:
   - **Baseline**: `perm = randperm_no_fixed(N)`; `get_correlation_a1b2(hiddens, perm)` → per-module correlation; average over samples → `base`.
   - **Fix feature 0**: subset `idx0 = (probes == 0)`; permute within idx0; `get_correlation_a1b2(hiddens[idx0], p0)` → `c0`; average → `corr0`.
   - **Fix feature 1**: same with `probes == 1` → `corr1`.
3. `norm0 = (corr0 - base) / (1 - base)`, `norm1 = (corr1 - base) / (1 - base)` (denom clipped to 1e-8 if ≤ 0).
4. `correlation_specialization_scalar(norm0, norm1)` → same global_diff formula on (norm0[0], norm0[1]) and (norm1[0], norm1[1]).

### Verification: correlation scalar with synthetic norm vectors

If norm0 = [1, 0] (M0 high when fix f0, M1 low) and norm1 = [0, 1] (flip for f1), we expect high specialization.

In [3]:
# norm_corr_fix0 = (M0, M1) when fixing feature 0; norm_corr_fix1 when fixing feature 1
# Specialized: M0 tracks f0, M1 tracks f1
norm0_spec = np.array([1.0, 0.0])
norm1_spec = np.array([0.0, 1.0])
corr_spec = correlation_specialization_scalar(norm0_spec, norm1_spec)
# Not specialized: both modules same for both
norm0_low = np.array([0.5, 0.5])
norm1_low = np.array([0.5, 0.5])
corr_low = correlation_specialization_scalar(norm0_low, norm1_low)
print("Specialized (M0→f0, M1→f1) norm vectors → scalar:", round(corr_spec, 4))
print("Not specialized (equal) → scalar:", round(corr_low, 4))

Specialized (M0→f0, M1→f1) norm vectors → scalar: 1.0
Not specialized (equal) → scalar: 0.0


### Interpretation and temporal detail (correlation)

- **Interpretation**: High correlation specialization means that when the **probed feature is held fixed** (e.g. only summer trials), one module’s activity is more **self-consistent** (higher within-module correlation) than the other, and this pattern **reverses** when the other feature is fixed. So the representation aligns with the feature (module–feature mapping). Low means both modules are similarly (in)consistent regardless of which feature is fixed.
- **Temporal detail**: Currently the npz stores **one block per phase**; the code **flattens all phases** and produces **one scalar per participant**. To get specialization **per phase**: call `compute_correlation_metric_a1b2` with `participant_data` where `hiddens_per_module` and `probes` are **one phase only** (e.g. `hiddens[phase]`, `probes[phase]`). You would need to ensure enough trials per phase for a stable correlation (and for both probe 0 and 1 to have ≥2 trials).

---
## 4. Ablation specialization

### Definition

- **Idea**: Use the **same 3-head probe** as retraining (only M0, only M1, both), but **no extra training** for the ablation metric: we only **evaluate** the trained probe. So we get acc[head, feature] from the probe that was trained for retraining. The **scalar formula is identical** to retraining: `retraining_specialization_scalar(acc[0,0], acc[1,0], acc[0,1], acc[1,1])`.
- **Difference from retraining**: Retraining measures “after we train a probe, how specialized is the decoding?” Ablation measures “with that same trained probe, how much does masking one module hurt for each feature?” So ablation is the **same scalar** but interpreted as **linear separability of features from each module’s representation** at readout time (no additional probe training).

### Data and computation

- **Data**: Same as retraining — trained wrapper with **probe readout already installed and trained** (from the retraining step in `03_functional_specialization.py`). If you run with `--no-retrain`, the script still trains the probe once for the ablation step.
- **Computation**: `compute_ablations_metric_a1b2(wrapper, loader, device)` → calls `eval_probe_readout_a1b2` (no training), then `retraining_specialization_scalar(acc[0,0], acc[1,0], acc[0,1], acc[1,1])` → `ablation_specialization`.
- **Verification**: Same as retraining: check scalar with synthetic acc matrix. Numerically, ablation and retraining can differ because retraining uses the **trained** probe accuracies, while conceptually ablation is “evaluate that same probe”; in the script they use the same probe, so the only difference is that retraining is computed right after training the probe, and ablation is the same evaluation — so **they are the same value** in the current pipeline (both use the same `acc` from `eval_probe_readout_a1b2`). So in practice **ablation_specialization == retraining_specialization** for a given participant unless you change the pipeline (e.g. train probe with different data or epochs).

### Interpretation and temporal detail (ablation)

- **Interpretation**: Same scale as retraining; high = one module’s representation (at the readout) is more informative for one feature and the other module for the other feature. Since the probe is the same as in retraining, the number is the same in the current script.
- **Temporal**: One value per participant (post training). Per-phase would require checkpoints and evaluating the probe (trained at that checkpoint or on that phase’s data) at each phase.

---
## 5. Recommended test cases

From the prior analysis: (1) **synthetic correlation test** — full correlation pipeline on engineered hiddens; (2) **integration check** — load saved specialization_metrics.json and verify consistency; (3) **task_routed vs shared** — compare mean specialization by input scheme (expect task_routed > shared).

### 5.1 Synthetic correlation test (full pipeline)

Build minimal `participant_data`: (a) **specialized** — M0 stable when probe=0, M1 stable when probe=1 → expect high correlation scalar; (b) **random** — no link to probe → expect low scalar. No training or npz required.

In [4]:
np.random.seed(42)
n_phase, n_trials, n_mod, h_size = 1, 200, 2, 10
probes = np.random.randint(0, 2, (n_phase, n_trials))
# Ensure both features have >= 2 trials for permutation
assert np.sum(probes == 0) >= 2 and np.sum(probes == 1) >= 2
inputs = np.zeros((n_phase, n_trials, 12))
inputs[:, :, 0] = 1  # dummy for stim_index

# (a) Specialized: M0 constant when probe=0, M1 constant when probe=1
hiddens_spec = np.random.randn(n_phase, n_trials, n_mod, h_size).astype(np.float32)
for p in range(n_phase):
    for t in range(n_trials):
        if probes[p, t] == 0:
            hiddens_spec[p, t, 0, :] = 1.0  # M0 constant for feature 0
        else:
            hiddens_spec[p, t, 1, :] = 1.0  # M1 constant for feature 1
out_spec = compute_correlation_metric_a1b2(
    {"hiddens_per_module": hiddens_spec, "probes": probes, "inputs": inputs},
    n_samples=20,
)
# (b) Random: no structure
hiddens_rand = np.random.randn(n_phase, n_trials, n_mod, h_size).astype(np.float32)
out_rand = compute_correlation_metric_a1b2(
    {"hiddens_per_module": hiddens_rand, "probes": probes, "inputs": inputs},
    n_samples=20,
)
print("Specialized (engineered M0↔f0, M1↔f1):", round(out_spec["correlation_specialization"], 4))
print("Random (no structure):", round(out_rand["correlation_specialization"], 4))
print("OK: specialized >> random" if out_spec["correlation_specialization"] > out_rand["correlation_specialization"] else "Check: expected specialized > random")

Specialized (engineered M0↔f0, M1↔f1): 0.5
Random (no structure): 0.0
OK: specialized >> random


/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/.venv/lib/python3.10/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


### 5.2 Integration check (from saved metrics)

Load `specialization_metrics.json` from a run folder (if present). Verify: (a) all three metrics in [0, 1]; (b) retraining ≈ ablation (same probe evaluation).

In [5]:
data_folder = project_root / "data"
sim_folder = data_folder / "simulations"
metrics_path = None
for run_dir in sim_folder.iterdir() if sim_folder.exists() else []:
    p = run_dir / "specialization_metrics.json"
    if p.exists():
        metrics_path = p
        break
if metrics_path is None:
    print("No specialization_metrics.json found in data/simulations/*. Skip integration check.")
else:
    import json
    with open(metrics_path) as f:
        data = json.load(f)
    run_id = data["run_id"]
    participants = data.get("participants", [])
    print(f"Run: {run_id}, participants: {len(participants)}")
    for key in ("retraining_specialization", "correlation_specialization", "ablation_specialization"):
        vals = [r[key] for r in participants if r.get(key) is not None]
        if not vals:
            print(f"  {key}: no values")
            continue
        in_range = all(0 <= v <= 1 for v in vals)
        print(f"  {key}: mean={np.mean(vals):.4f}, in [0,1]: {in_range}, n={len(vals)}")
    retrain = [r["retraining_specialization"] for r in participants if r.get("retraining_specialization") is not None]
    ablate = [r["ablation_specialization"] for r in participants if r.get("ablation_specialization") is not None]
    if retrain and ablate and len(retrain) == len(ablate):
        diff = np.abs(np.array(retrain) - np.array(ablate))
        print(f"  |retraining - ablation|: max={np.max(diff):.6f}, mean={np.mean(diff):.6f}")
        print("  OK: retraining ≈ ablation" if np.max(diff) < 0.01 else "  Check: retraining and ablation should match (same probe)")

No specialization_metrics.json found in data/simulations/*. Skip integration check.


### 5.3 Meaningful test: task_routed vs shared

For two-module networks with communication, **task_routed** input (A→M0, B→M1) should yield higher specialization than **shared** (same sparsity). Load the extended CSV and compare mean specialization by `input_scheme`.

In [6]:
import pandas as pd
_data_folder = project_root / "data"
extended_path = _data_folder / "analysis" / "functional_specialization_extended.csv"
if not extended_path.exists():
    print("functional_specialization_extended.csv not found. Skip task_routed vs shared.")
else:
    df = pd.read_csv(extended_path)
    if "input_scheme" not in df.columns:
        print("No input_scheme column in CSV. Skip.")
    else:
        # Compare same sparsity_label: task_routed vs shared
        for metric in ["retraining_specialization", "correlation_specialization", "ablation_specialization"]:
            if metric not in df.columns:
                continue
            sub = df.dropna(subset=[metric])
            if sub.empty:
                continue
            by_scheme = sub.groupby("input_scheme", dropna=True)[metric].agg(["mean", "sem", "count"])
            print(metric)
            print(by_scheme.to_string())
            if "task_routed" in by_scheme.index and "shared" in by_scheme.index:
                tr = by_scheme.loc["task_routed", "mean"]
                sh = by_scheme.loc["shared", "mean"]
                print(f"  task_routed > shared: {tr > sh} (task_routed={tr:.4f}, shared={sh:.4f})")
            print()

retraining_specialization
                  mean       sem  count
input_scheme                           
shared        0.069471  0.010875     15
task_routed   0.099982  0.016676     14
  task_routed > shared: True (task_routed=0.1000, shared=0.0695)

correlation_specialization
                  mean      sem  count
input_scheme                          
shared        0.000000  0.00000     90
task_routed   2.104568  1.68537     89
  task_routed > shared: True (task_routed=2.1046, shared=0.0000)

ablation_specialization
                  mean       sem  count
input_scheme                           
shared        0.069428  0.010731     15
task_routed   0.098524  0.016068     14
  task_routed > shared: True (task_routed=0.0985, shared=0.0694)



---
## 6. Summary: what to watch for and how to verify

### Data checklist (per participant)

| Metric | Required data | Where it comes from |
|--------|----------------|----------------------|
| Retraining | Trained wrapper (state_*.pt), DataLoader A1+B+A2 with `input`, `label_x`, `label_y`, `feature_probe` | `run_loader`, `get_datasets`, `CreateParticipantDataset` |
| Correlation | `hiddens_per_module`, `probes`, `inputs` in sim_*.npz; shapes (n_phase, n_trials, …) | Simulation must save these (schedule saves them when `rnn_extra` is set) |
| Ablation | Same as retraining; probe must be trained first | Same loader as retraining |

### Verification checklist

1. **Scalar formula**: Run synthetic accs (0.9, 0.1, 0.1, 0.9) → scalar ≈ 0.8; (0.5, 0.5, 0.5, 0.5) → 0. Run synthetic norm vectors for correlation → high when [1,0] vs [0,1].
2. **Probe indexing**: In retraining/ablation, head 0 = only M0 (mask M1), head 1 = only M1 (mask M0). So acc[0, f] = only-M0 on feature f.
3. **Correlation baseline**: Baseline uses a permutation with **no fixed points** so it’s not trivially 1. Fix-feature subsets must have ≥2 trials for a valid permutation.
4. **npz shapes**: If `hiddens_per_module` is (n_phase, n_trials, n_mod, h), correlation code flattens to (N, n_mod, h). If your simulation saves different shapes (e.g. no phase dim), adapt the flatten step.

### Why numbers might look “off”

- **Probe training**: Only 5 epochs; different `n_epochs` or `lr` can change retraining (and thus ablation) a lot. Try increasing epochs or checking loss curve of the probe.
- **Loader content**: Retraining uses **A1+B+A2** concatenated. If B is much larger than A1/A2, the probe may be biased toward B’s feature; balance or phase-weighted sampling could help.
- **Correlation**: `n_samples=10` is small; increase for stability. If one feature has very few trials (e.g. probe==1 rare), fix-feature-1 correlation is noisy.
- **Ablation vs retraining**: They should match in the current script; if not, check that the same `acc` is used and that you didn’t re-train the probe between the two.
- **Per-module hiddens**: Correlation needs **per-module** hidden states. If the simulation only saved concatenated hiddens, you must split with `_split_hiddens_by_module` when writing the npz (or when loading, if you have n_modules and hidden_size).

### Temporal detail over training

- **Current**: One scalar per participant per metric (post full A1–B–A2 training).
- **Per-phase**: For **correlation**: run the metric on `hiddens_per_module[phase]` and `probes[phase]` per phase. For **retraining/ablation**: need checkpoints and (optionally) phase-specific loaders and/or probe training per checkpoint.